In [3]:
"""Generate one reproducible monodisperse aggregate."""
from fracval import FracVALConfig, generate, save_bundle, plot_3d

config = FracVALConfig(
    n=100,
    df=1.79,
    kf=1.40,
    rp_g=15.0,
    rp_gstd=1.0,
    seed=12345,
)

aggregate = generate(config)
print(f"backend: {aggregate.backend}")
print(f"seed: {aggregate.seed}")
print(f"particles: {aggregate.n}")
print(f"radius of gyration: {aggregate.radius_of_gyration:.6g}")
print(f"bounding radius: {aggregate.bounding_radius:.6g}")

written = save_bundle(aggregate, "example_output/basic", stem="monodisperse")
for kind, path in written.items():
    print(f"{kind}: {path}")

plot_3d(aggregate, mode="spheres")

backend: extension
seed: 12345
particles: 100
radius of gyration: 162.86
bounding radius: 306.618
dat: example_output/basic/monodisperse.dat
csv: example_output/basic/monodisperse.csv
contacts: example_output/basic/monodisperse.contacts.csv
json: example_output/basic/monodisperse.json


In [4]:
"""Generate a polydisperse aggregate and display it interactively."""
from fracval import FracVALConfig, ViewerAppearance, generate, plot_3d

aggregate = generate(FracVALConfig(
    n=100,
    df=1.68,
    kf=0.98,
    rp_g=15.0,
    rp_gstd=2.0,
    seed=67890,
))

appearance = ViewerAppearance(
    color_mode="radius",
    colorscale="Viridis",
    opacity=0.95,
    shininess=0.55,
    show_axes=False,
    show_colorbar=True,
)

figure = plot_3d(aggregate, mode="spheres", appearance=appearance)
figure.show()

In [10]:
"""Compare the intended-contact overlap modes."""
from fracval import FracVALConfig, generate

base = FracVALConfig(n=50, seed=24680, max_attempts=500)

cases = {
    "none": base.with_updates(overlap_mode="none"),
    "fixed_5_percent": base.with_updates(
        overlap_mode="fixed", overlap_fraction=0.05
    ),
    "statistical": base.with_updates(
        overlap_mode="statistical",
        overlap_mean=0.05,
        overlap_std=0.02,
        overlap_max=0.12,
    ),
}

for name, config in cases.items():
    aggregate = generate(config)
    print(
        f"{name:16s} contacts={aggregate.contact_count:3d} "
        f"mean={100*aggregate.mean_contact_overlap:6.3f}% "
        f"std={100*aggregate.std_contact_overlap:6.3f}% "
        f"max={100*aggregate.max_contact_overlap:6.3f}%"
    )
    plot_3d(aggregate).show()

none             contacts= 49 mean= 0.000% std= 0.000% max= 0.000%


fixed_5_percent  contacts= 49 mean= 5.000% std= 0.000% max= 5.000%


statistical      contacts= 49 mean= 4.612% std= 1.877% max= 8.740%


In [3]:
"""Generate a deterministic batch and save each aggregate as a bundle."""
from pathlib import Path

from fracval import FracVALConfig, iter_generate_batch, save_bundle, plot_3d

output = Path("example_output/batch")
config = FracVALConfig(n=100, df=1.79, kf=1.40, seed=20260902)

for index, aggregate in enumerate(iter_generate_batch(10, config), start=1):
    stem = f"aggregate_{index:04d}_seed_{aggregate.seed}"
    save_bundle(aggregate, output, stem=stem)
    print(index, aggregate.seed, aggregate.radius_of_gyration)
    plot_3d(aggregate).show()

1 1845927062 162.86029657931442


2 1251157179 162.86007655993492


3 792082136 162.86039184156522


4 364004586 162.8605384646041


5 334651691 162.86033154900352


6 811202724 162.86033872410934


7 1638425206 162.86033354558475


8 1040888733 162.86055532720968


9 1040754033 162.86028685679446


10 1173531854 162.8603590742458


In [7]:
from fracval import available_backends
print(available_backends())

['extension', 'executable']


In [6]:
"""Small parameter sweep suitable as a template for a research workflow."""
import csv
from pathlib import Path

from fracval import FracVALConfig, generate

rows = []
base_seed = 10000
for i, df in enumerate((1.6, 1.8, 2.0)):
    for j, kf in enumerate((0.9, 1.2, 1.5)):
        config = FracVALConfig(
            n=100,
            df=df,
            kf=kf,
            rp_g=15.0,
            rp_gstd=1.0,
            seed=base_seed + 100 * i + j,
            max_attempts=500,
        )
        print(df, kf)
        aggregate = generate(config)
        rows.append({
            "df": df,
            "kf": kf,
            "seed": aggregate.seed,
            "radius_of_gyration": aggregate.radius_of_gyration,
            "bounding_radius": aggregate.bounding_radius,
            "attempts": aggregate.attempts,
        })
        print(rows[-1])

out = Path("example_output/parameter_sweep.csv")
out.parent.mkdir(parents=True, exist_ok=True)
with out.open("w", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=rows[0])
    writer.writeheader()
    writer.writerows(rows)
print(f"wrote {out}")

1.6 0.9


GenerationError: FracVAL failed to build an aggregate in 500 attempts